# HQCNN for Medical Image Classification
## Hybrid Quantum–Classical CNN Demonstration

**Vilnius University · Faculty of Mathematics and Informatics**

This notebook demonstrates the core models from the HQCNN benchmarking study:
- **Model A** — Linear baseline
- **Model B** — Classical bottleneck (nonlinear MLP)
- **B-linear** — Capacity-matched linear bottleneck
- **Model C** — Hybrid quantum head (8-qubit VQC)

All models share the same pretrained ResNet-18 backbone.
Only the classifier head varies, enabling a fair comparison.

---
**Dataset:** DermaMNIST (7-class skin lesion classification, 28×28 images)  
**Primary metric:** Macro F1 (class-imbalanced dataset)  
**Quantum framework:** PennyLane + PyTorch

In [ ]:
# Install dependencies
!pip install pennylane pennylane-lightning medmnist scikit-learn -q

In [ ]:
# Clone the repository
!git clone https://github.com/Sasan-Ansarian/hqcnn-medical-imaging.git
%cd hqcnn-medical-imaging

## 1. Setup and Data Loading

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import medmnist
from medmnist import DermaMNIST
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from sklearn.metrics import f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

import pennylane as qml
print(f'PennyLane: {qml.__version__}')

In [ ]:
# Load DermaMNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_dataset = DermaMNIST(split='train', transform=transform, download=True)
test_dataset  = DermaMNIST(split='test',  transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=128, shuffle=False, num_workers=2)

NUM_CLASSES = 7
print(f'Train samples: {len(train_dataset)}')
print(f'Test  samples: {len(test_dataset)}')
print(f'Classes: {NUM_CLASSES}')

## 2. Build Models

In [ ]:
from src.models.factory import build_model

# Model A — linear baseline (frozen backbone)
model_a = build_model({'name': 'resnet18_small_baseline_frozen'}, NUM_CLASSES).to(device)

# Model B — classical bottleneck 512→32→8 (frozen backbone)
model_b = build_model({'name': 'resnet18_bottleneck_32_8_frozen'}, NUM_CLASSES).to(device)

# B-linear — capacity-matched linear bottleneck (end-to-end)
model_blin = build_model({'name': 'resnet18_bottleneck_linear_16_4_e2e'}, NUM_CLASSES).to(device)

# Model C — 8-qubit hybrid quantum head 512→32→8 (frozen backbone)
model_c = build_model({'name': 'resnet18_quantum_32_8_frozen'}, NUM_CLASSES).to(device)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Model A  trainable params: {count_params(model_a):,}')
print(f'Model B  trainable params: {count_params(model_b):,}')
print(f'B-linear trainable params: {count_params(model_blin):,}')
print(f'Model C  trainable params: {count_params(model_c):,}')

## 3. Quantum Circuit Inspection

In [ ]:
# Visualise the quantum circuit
import pennylane as qml

n_qubits = 8
n_layers = 2
dev = qml.device('default.qubit', wires=n_qubits)

@qml.qnode(dev)
def demo_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation='Y')
    for layer in range(n_layers):
        for i in range(n_qubits):
            qml.RY(weights[layer, i], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

# Draw the circuit
inputs  = np.zeros(n_qubits)
weights = np.zeros((n_layers, n_qubits))
print(qml.draw(demo_circuit)(inputs, weights))

## 4. Quick Training Demo (5 epochs)

In [ ]:
from src.train.engine import train_one_epoch, evaluate

# Class weights for DermaMNIST (inverse frequency)
class_counts = torch.zeros(NUM_CLASSES)
for _, labels in train_loader:
    labels = labels.squeeze()
    for c in range(NUM_CLASSES):
        class_counts[c] += (labels == c).sum()
class_weights = (1.0 / class_counts).to(device)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES

criterion = nn.CrossEntropyLoss(weight=class_weights)

def run_demo(model, name, epochs=5, lr=5e-4):
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=0.01
    )
    print(f'\n--- {name} ---')
    for epoch in range(1, epochs + 1):
        train_stats = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_stats   = evaluate(model, test_loader, criterion, device)
        print(f'  Epoch {epoch}/{epochs} | '
              f'loss={train_stats.loss:.4f} | '
              f'macro_F1={val_stats.macro_f1:.4f} | '
              f'grad_norm={train_stats.grad_norm_mean:.3f} '
              f'(var={train_stats.grad_norm_var:.3f})')
    return val_stats

# Demo: Model A and Model C (5 epochs each)
# Note: full results require 20 epochs + multiple seeds as in the paper
results_a = run_demo(model_a, 'Model A (linear baseline)')
results_c = run_demo(model_c, 'Model C (quantum head)')

## 5. Results Summary

In [ ]:
print('\n=== 5-epoch demo results ===')
print(f'Model A | Macro F1: {results_a.macro_f1:.4f} | Accuracy: {results_a.accuracy:.4f}')
print(f'Model C | Macro F1: {results_c.macro_f1:.4f} | Accuracy: {results_c.accuracy:.4f}')
print()
print('=== Full 20-epoch results from the paper (3 seeds, frozen backbone) ===')
print('Model A | Macro F1: 0.472 ± 0.012 | Accuracy: 0.739 ± 0.002')
print('Model B | Macro F1: 0.390 ± 0.024 | Accuracy: 0.731 ± 0.001')
print('Model C | Macro F1: 0.317 ± 0.014 | Accuracy: 0.728 ± 0.006')
print()
print('Key finding: Classical baseline consistently outperforms quantum head')
print('on macro F1. A narrow transition regime exists at 8→2 compression')
print('+ 20% data, but it does not generalise across datasets.')

## 6. Running Full Experiments on HPC

The complete 27-experiment programme was executed on the VU MIF HPC infrastructure.
To reproduce any experiment:

```bash
# Single experiment
python scripts/run_experiment.py --config configs/chapter7/exp3_interface.yaml

# Multi-seed array job (SLURM)
sbatch slurm/train_array.sbatch
```

See `docs/report.pdf` for the full experimental results and analysis.